# [실습6] 양극재 결정구조 예측 딥러닝 모델 성능 향상을 위한 다양한 기법
---

## 실습 목표

- 딥러닝 학습 시 적용할 수 있는 심화 기법들을 이해하고 구현합니다.
- 기법들의 적용 유무에 따른 성능을 비교해봅니다. 각각 토이 데이터셋과 실습 데이터셋에서 검증합니다.

---

## 실습 목차

1. **토이 데이터셋 불러오기 및 전처리 적용** 

2. **학습 테스트 결과 추적** 

3. **조기 종료(Early Stopping)**

4. **그래픽카드 사용하기** 

5. **데이터 증강** 

6. **가중치 규제**

7. **드롭아웃(Dropout)**

8. **실습 데이터 불러오기 및 딥러닝 기법 적용** 

9. **AutoML**
---

## 실습 개요

이번 실습에서는 머신러닝 분류 모델들을 구현하고 실습 데이터에 어떻게 적용될 수 있는지 확인합니다.

---

## 1. 토이 데이터셋 불러오기 및 전처리 적용
---
기존에 다뤘던 전처리를 모두 적용해 학습 & 테스트 데이터를 구분하겠습니다.

In [ ]:
## 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

### 1.1 토이 데이터셋 불러오기
타이타닉 데이터셋을 불러옵니다.

In [ ]:
df = pd.read_csv('./data/titanic_processed.csv') # df: dataframe의 줄임말
df.drop(columns = df.columns[0], axis = 1, inplace= True) # 데이터 순서를 표현하는 첫 열을 제거합니다.
display(df)

해당 승객의 생존 여부를 정답 데이터로 지정하겠습니다. 나머지 데이터를 입력 데이터로 지정합니다.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['pclass', 'age', 'sibsp', 'parch', 'fare', 'gender', 'embarked']]
y = df['survived']

display(X)
display(y)


---
### 1.2 원-핫 인코딩(One-Hot Encoding)
범주형 데이터에 대해 원-핫 인코딩을 적용해보겠습니다.

pclass에 원-핫 인코딩 적용합니다.

### [TODO] 위 설명을 참고하여 ______를 수정해보세요.

In [ ]:
dummies = pd.get_dummies(X['pclass']).astype('int')
dummies.columns = ['1class', '2class', '3class']

X.drop('_______', axis=1, inplace=True)
X = X.join(dummies)

gender에 원-핫 인코딩 적용합니다.

In [ ]:
dummies = pd.get_dummies(X['gender']).astype('int')
dummies.columns = ['female', 'male']
X.drop('gender', axis=1, inplace=True)
X = X.join(dummies)

embarked에 원-핫 인코딩을 적용합니다.

In [ ]:
dummies = pd.get_dummies(X['embarked']).astype('int')
dummies.columns = ['S', 'C', 'Q']

X.drop('embarked', axis=1, inplace=True)
X = X.join(dummies)

---
### 1.3 정규화 & 표준화
원-핫 인코딩이 완료된 토이 데이터셋에 정규화와 표준화를 진행합니다. 

In [ ]:
from sklearn.preprocessing import StandardScaler
std_scaler = StandardScaler()
std_scaler.fit(X)
scaled_X = std_scaler.transform(X)
X = pd.DataFrame(scaled_X, columns=X.columns, index=list(X.index.values))
display(X)

---
### 1.4 학습 & 테스트 데이터 구분

In [ ]:
from keras.utils import to_categorical

random_seed = 42
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size = 0.3, random_state = random_seed)
Y_train = to_categorical(Y_train)
Y_test_cat = to_categorical(Y_test)

print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test_cat.shape)

---
## 2. 학습 테스트 결과 추적
MLP 모델을 구현하고, 데이터를 학습하는 과정을 추적합니다.

---
### 2.1 MLP 모델 구성
우선 MLP 모델을 만듭니다.


### [TODO] 적합한 모델의 출력 크기를 고려하여 ______를 수정해보세요.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input

tf.random.set_seed(random_seed)
np.random.seed(random_seed)

MLP_model = tf.keras.Sequential([
    Input(shape=X_train.shape[1]),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dense(32, activation = 'relu'),
    tf.keras.layers.Dense(__, activation='softmax')
])

---
### 2.2 학습 방법 설정
손실 함수와 최적화함수를 설정합니다.

In [ ]:
lr = 0.0001
MLP_model.compile(loss = 'categorical_crossentropy',
              optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
              metrics=['accuracy']
)

---
### 2.3 모델 학습
테스트 데이터를 검증 데이터로 사용하겠습니다.

In [ ]:
epoch = 300
history = MLP_model.fit(X_train, Y_train, epochs = epoch, 
                        batch_size = 16, verbose = 1, validation_data=(X_test, Y_test_cat))
print(history)

---
### 2.4 학습 결과 시각화
학습 & 테스트 결과를 시각화합니다. 먼저 손실량(loss)에 대해 시각화합니다.

In [ ]:
plt.title('Loss / Mean Squared Error')
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()
plt.show()

정답률에 대해서도 시각화합니다.

In [ ]:
plt.title('Model Accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.plot(history.history['accuracy'], label='train accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epochs')
plt.legend()
plt.show()

학습이 일정 에폭 이상 진행되면 더이상 검증 성능이 개선되지 않는 것을 확인할 수 있습니다.

---
## 3. 조기 종료(Early Stopping)
학습이 더이상 개선되지 않는다고 판단되면 학습 과정을 조기 종료시킵니다.

---
### 3.1 조기 종료 설정
keras 라이브러리에 내장된 EarlyStopping 콜백 함수를 사용하겠습니다.

검증 손실량이 30 에폭 이상 개선되지 않으면 과정을 조기 종료시키겠습니다.

In [ ]:
from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor = 'val_loss', min_delta = 0, 
                               patience = 30, mode = 'auto',
                               restore_best_weights=True)

---
### 3.2 모델 학습 및 결과 시각화

In [ ]:
MLP_model = tf.keras.Sequential([
    Input(shape=X_train.shape[1]),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dense(32, activation = 'relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])

In [ ]:
lr = 0.0001
MLP_model.compile(loss = 'categorical_crossentropy',
              optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
              metrics=['accuracy']
)

In [ ]:
history = MLP_model.fit(X_train, Y_train, 
                        epochs = epoch, 
                        batch_size = 16, 
                        verbose = 1, 
                        validation_data=(X_test, Y_test_cat),
                        callbacks = [early_stopping])

In [ ]:
plt.title('Loss')
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()
plt.show()

In [ ]:
plt.title('Model Accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.plot(history.history['accuracy'], label='train accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epochs')
plt.legend()
plt.show()

---
## 4. 그래픽카드 사용하기

큰 데이터셋을 학습할 때는 학습 시간이 매우 오래걸립니다.

이때, 적절한 그래픽카드를 사용하면 학습 시간을 크게 단축할 수 있습니다.

---
### 4.1 그래픽카드 정보 확인


In [ ]:
from tensorflow.python.client import device_lib

print(device_lib.list_local_devices())

CPU와 그래픽카드(GPU)의 정보를 확인할 수 있습니다.

첫번째 GPU는 GPU:0으로 표기됩니다.

---
### 4.2 그래픽카드를 이용한 학습
정확한 학습 속도 비교를 위해 Early Stopping을 적용하지 않고 학습을 진행해보겠습니다.

각각 CPU와 GPU를 이용해 학습을 진행합니다.

In [ ]:
with tf.device("/device:CPU:0"):
    MLP_model = tf.keras.Sequential([
        Input(shape=X_train.shape[1]),
        tf.keras.layers.Dense(64, activation = 'relu'),
        tf.keras.layers.Dense(128, activation = 'relu'),
        tf.keras.layers.Dense(512, activation = 'relu'),
        tf.keras.layers.Dense(2048, activation = 'relu'),
        tf.keras.layers.Dense(1024, activation = 'relu'),
        tf.keras.layers.Dense(512, activation = 'relu'),
        tf.keras.layers.Dense(128, activation = 'relu'),
        tf.keras.layers.Dense(2, activation='softmax')
    ])
    lr = 0.0001
    MLP_model.compile(loss = 'categorical_crossentropy',
                  optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
                  metrics=['accuracy']
    )

먼저 CPU를 이용해 300 에폭을 학습시키겠습니다.

In [ ]:
import time

start_time = time.time()
with tf.device("/device:CPU:0"):
    history = MLP_model.fit(X_train, Y_train, 
                            epochs = epoch, 
                            batch_size = 16, 
                            verbose = 0, 
                            validation_data=(X_test, Y_test_cat))
end_time = time.time()


### [TODO] 학습 시간을 출력하기 위해 ______를 수정해보세요.

In [ ]:
print("{} 에폭을 학습하는데 총 {:.3f} 초가 걸렸습니다.".format(epoch, _______-______))

다음 GPU를 이용해 300 에폭을 학습시키겠습니다.

In [ ]:
with tf.device("/device:GPU:0"):
    MLP_model = tf.keras.Sequential([
        Input(shape=X_train.shape[1]),
        tf.keras.layers.Dense(64, activation = 'relu'),
        tf.keras.layers.Dense(128, activation = 'relu'),
        tf.keras.layers.Dense(512, activation = 'relu'),
        tf.keras.layers.Dense(2048, activation = 'relu'),
        tf.keras.layers.Dense(1024, activation = 'relu'),
        tf.keras.layers.Dense(512, activation = 'relu'),
        tf.keras.layers.Dense(128, activation = 'relu'),
        tf.keras.layers.Dense(2, activation='softmax')
    ])
    lr = 0.0001
    MLP_model.compile(loss = 'categorical_crossentropy',
                  optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
                  metrics=['accuracy']
    )

In [ ]:
start_time = time.time()
with tf.device("/device:GPU:0"):
    history = MLP_model.fit(X_train, Y_train, 
                            epochs = epoch, 
                            batch_size = 16, 
                            verbose = 0, 
                            validation_data=(X_test, Y_test_cat))
end_time = time.time()

In [ ]:
print("{} 에폭을 학습하는데 총 {:.3f} 초가 걸렸습니다.".format(epoch, end_time-start_time))

학습 시간에 차이가 있는 것을 확인할 수 있습니다.

일반적으로, 여러 가지 이유가 학습 속도에 영향을 미칩니다.

1. 데이터의 크기
2. 배치 사이즈
3. 한 데이터 샘플의 크기 (숫자, 이미지, 음성 등)
4. 모델의 파라미터 사이즈

데이터 셋의 크기가 작으면 배치 사이즈가 일반적으로 작습니다. 또, 작은 숫자 데이터를 학습시킬 경우, 학습 과정이 매우 계산효율적으로 이뤄집니다.

그래픽카드의 병렬 계산 성능으로 얻을 수 있는 이득이 매우 작을 경우, GPU에서 학습하는 것보다 CPU에서 학습하는 것이 더 빠를수도 있습니다.

이는 CPU-GPU 사이의 정보 전달 시간이 소요되기 때문입니다. 이러한 메모리 복사에 의해 소요되는 시간을 CPU-GPU 병목(Bottleneck)이라고 부릅니다.

---
## 5. 데이터 증강
---
### 5.1 데이터 증강 없는 모델 성능 평가

In [ ]:
model = tf.keras.Sequential([
    Input(shape=X_train.shape[1]),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dense(32, activation = 'relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])
lr = 0.0001
model.compile(loss = 'categorical_crossentropy',
              optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
              metrics=['accuracy']
)
early_stopping = EarlyStopping(monitor = 'val_loss', min_delta = 0, 
                               patience = 30, mode = 'auto',
                               restore_best_weights=True)

In [ ]:

history = model.fit(X_train, Y_train, 
                        epochs = epoch, 
                        batch_size = 16, 
                        verbose = 0, 
                        validation_data=(X_test, Y_test_cat),
                        callbacks=[early_stopping])

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluation(prediction, Y_test, model_name="모델 이름"):
    print(model_name, "의 정답률은 %.2f%% 입니다." % (accuracy_score(prediction, Y_test) * 100))
    print(model_name, "의 정밀도는 %.2f 입니다." % precision_score(Y_test, prediction))
    print(model_name, "의 재현율은 %.2f 입니다." % recall_score(Y_test, prediction))
    print(model_name, "의 F1 점수는 %.2f 입니다." % f1_score(Y_test, prediction))
    print(model_name, "의 ROC-AUC 점수는 %.2f 입니다." % roc_auc_score(Y_test, prediction))
    return accuracy_score(prediction, Y_test) * 100, roc_auc_score(Y_test, prediction)

In [ ]:
pred = model.predict(X_test)
pred_classes=np.argmax(pred,axis=1)
acc, auc = evaluation(pred_classes, Y_test, model_name="MLP")

추후 시각화를 위해 결과를 저장합니다.

In [ ]:
acc_dict = {}
auc_dict = {}
acc_dict['Baseline'] = acc
auc_dict['Baseline'] = auc

---
### 5.2 데이터 증강을 적용한 모델 성능 평가
데이터 증강을 적용해보겠습니다. 학습 데이터에 가우시안 노이즈를 추가하는 것으로 간단한 데이터 증강을 할 수 있습니다.

학습 데이터에 노이즈를 직접 추가할 경우, 노이즈가 추가된 데이터가 되기 때문에 잘못된 데이터 증강 방법입니다.

매번 새로운 노이즈가 추가되도록 하여 데이터가 증가된 것 처럼 적용하는 것이 핵심입니다.

모델에 노이즈 레이어를 추가함으로써 쉽게 적용할 수 있습니다.

In [ ]:
model_aug = tf.keras.Sequential([
    Input(shape=X_train.shape[1]),
    tf.keras.layers.GaussianNoise(stddev = 0.1),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dense(32, activation = 'relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])
lr = 0.0001
model_aug.compile(loss = 'categorical_crossentropy',
              optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
              metrics=['accuracy']
)
early_stopping = EarlyStopping(monitor = 'val_loss', min_delta = 0, 
                               patience = 30, mode = 'auto',
                               restore_best_weights=True)

In [ ]:
history = model_aug.fit(X_train, Y_train, 
                        epochs = epoch, 
                        batch_size = 16, 
                        verbose = 0, 
                        validation_data=(X_test, Y_test_cat),
                        callbacks=[early_stopping])

In [ ]:
pred = model_aug.predict(X_test)
pred_classes=np.argmax(pred,axis=1)
acc, auc = evaluation(pred_classes, Y_test, model_name="데이터 증강 MLP")

In [ ]:
acc_dict['Augmentation'] = acc
auc_dict['Augmentation'] = auc



---
## 6. 가중치 규제
---
### 6.1 가중치 규제 적용
L2 regularizer를 적용해보겠습니다.

In [ ]:
model_reg = tf.keras.Sequential([
    Input(shape=X_train.shape[1]),
    tf.keras.layers.Dense(64, activation = 'relu', kernel_regularizer=tf.keras.regularizers.L2(0.001)),
    tf.keras.layers.Dense(64, activation = 'relu', kernel_regularizer=tf.keras.regularizers.L2(0.001)),
    tf.keras.layers.Dense(32, activation = 'relu', kernel_regularizer=tf.keras.regularizers.L2(0.001)),
    tf.keras.layers.Dense(2, activation='softmax')
])

In [ ]:
lr = 0.0001
model_reg.compile(loss = 'categorical_crossentropy',
              optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
              metrics=['accuracy']
)
early_stopping = EarlyStopping(monitor = 'val_loss', min_delta = 0, 
                               patience = 30, mode = 'auto',
                               restore_best_weights=True)

In [ ]:
history = model_reg.fit(X_train, Y_train, 
                        epochs = epoch, 
                        batch_size = 16, 
                        verbose = 0, 
                        validation_data=(X_test, Y_test_cat),
                        callbacks=[early_stopping])

In [ ]:
pred = model_reg.predict(X_test)
pred_classes=np.argmax(pred,axis=1)
acc, auc = evaluation(pred_classes, Y_test, model_name="가중치 규제 MLP")

In [ ]:
acc_dict['Regularization'] = acc
auc_dict['Regularization'] = auc

---
## 7. 드롭아웃(Dropout)
---
### 7.1 드롭아웃 적용
모델에 드롭아웃을 적용해보겠습니다. 각 은닉층의 20%의 뉴런에 드롭아웃을 적용합니다.


### [TODO] 위 설명을 참고하여 ______를 수정해보세요.

In [ ]:
model_drop = tf.keras.Sequential([
    Input(shape=X_train.shape[1]),
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dropout(rate=___), 
    tf.keras.layers.Dense(64, activation = 'relu'),
    tf.keras.layers.Dropout(rate=___), 
    tf.keras.layers.Dense(32, activation = 'relu'),
    tf.keras.layers.Dropout(rate=___), 
    tf.keras.layers.Dense(2, activation='softmax')
])

In [ ]:
lr = 0.0001
model_drop.compile(loss = 'categorical_crossentropy',
              optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
              metrics=['accuracy']
)
early_stopping = EarlyStopping(monitor = 'val_loss', min_delta = 0, 
                               patience = 30, mode = 'auto',
                               restore_best_weights=True)

In [ ]:
history = model_drop.fit(X_train, Y_train, 
                        epochs = epoch, 
                        batch_size = 16, 
                        verbose = 0, 
                        validation_data=(X_test, Y_test_cat),
                        callbacks=[early_stopping])

In [ ]:
pred = model_drop.predict(X_test)
pred_classes=np.argmax(pred,axis=1)
acc, auc = evaluation(pred_classes, Y_test, model_name="드롭아웃 MLP")

In [ ]:
acc_dict['Dropout'] = acc
auc_dict['Dropout'] = auc

In [ ]:
bar = plt.bar(acc_dict.keys(), acc_dict.values())
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2.0, height, '%.2f' % height, ha='center', va='bottom', size = 12)
plt.ylim(0, 110)
plt.show()   

In [ ]:
bar = plt.bar(auc_dict.keys(), auc_dict.values())
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2.0, height, '%.3f' % height, ha='center', va='bottom', size = 12)
plt.ylim(0, 1.1)
plt.show()   


---
## 8. 실습 데이터 불러오기 및 딥러닝 기법 적용
---
### 8.1 실습 데이터 불러오기
모든 전처리가 끝난 실습데이터를 불러오겠습니다.

In [ ]:
df = pd.read_csv('./data/processed.csv') # df: dataframe의 줄임말
df.drop(columns = df.columns[0], axis = 1, inplace= True) # 데이터 순서를 표현하는 첫 열을 제거합니다.
display(df)

---
### 8.2 학습 & 테스트 데이터 구분
7:3 의 비율로 학습 데이터와 테스트 데이터를 구분합니다.

이때 Crystal System 데이터를 정답 데이터로 분리합니다.

In [ ]:
target_col = 'Crystal System'

y = df[target_col]
y = y.astype('category').cat.codes
X = df.drop(target_col, axis=1)
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.3, random_state=random_seed)

Y_train = to_categorical(Y_train)
Y_test_cat = to_categorical(Y_test)

print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

---
### 8.3 MLP 모델 구성
은닉층 5개를 사용하는 MLP 모델을 구성하겠습니다.

아무 기법도 적용하지 않은 베이스라인 모델 먼저 학습하겠습니다.


### [TODO] 적합한 모델 출력층의 크기를 고려하여 ______를 수정해보세요.

In [ ]:
model = tf.keras.Sequential([
    Input(shape=X_train.shape[1]),
    tf.keras.layers.Dense(1024, activation = 'relu'),
    tf.keras.layers.Dense(512, activation = 'relu'),
    tf.keras.layers.Dense(256, activation = 'relu'),
    tf.keras.layers.Dense(128, activation = 'relu'),
    tf.keras.layers.Dense(__, activation='softmax')
])

lr = 0.0001
model.compile(loss = 'categorical_crossentropy',
              optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
              metrics=['accuracy']
)
early_stopping = EarlyStopping(monitor = 'val_loss', min_delta = 0, 
                               patience = 150, mode = 'auto',
                               restore_best_weights=True)

---
### 8.4 모델 학습

In [ ]:
epoch = 1000
history = model.fit(X_train, Y_train, 
                        epochs = epoch, 
                        batch_size = 16, 
                        verbose = 0, 
                        validation_data=(X_test, Y_test_cat),
                        callbacks=[early_stopping])

In [ ]:
plt.title('Loss')
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()
plt.show()

In [ ]:
def evaluation(prediction, Y_test, model_name="모델 이름"):
    print(model_name, "의 정답률은 %.2f%% 입니다." % (accuracy_score(prediction, Y_test) * 100))
    print(model_name, "의 정밀도는 %.2f 입니다." % precision_score(Y_test, prediction, average='weighted'))
    print(model_name, "의 재현율은 %.2f 입니다." % recall_score(Y_test, prediction, average='weighted'))
    print(model_name, "의 F1 점수는 %.2f 입니다." % f1_score(Y_test, prediction, average='weighted'))
    return accuracy_score(prediction, Y_test) * 100

In [ ]:
pred = model.predict(X_test)
pred_classes = np.argmax(pred,axis=1)
acc = evaluation(pred_classes, Y_test, model_name="MLP")

In [ ]:
acc_dict = {}
acc_dict['Baseline'] = acc

---
### 8.5 딥러닝 기법 적용
데이터 증강, 가중치 규제, 드롭아웃을 적용시키겠습니다.


### [TODO] L2 가중치 규제를 적용하기 위해 ______를 수정해보세요.


In [ ]:
model_adv = tf.keras.Sequential([
    Input(shape=X_train.shape[1]),
    tf.keras.layers.GaussianNoise(stddev = 0.1),
    tf.keras.layers.Dense(1024, activation = 'relu', kernel_regularizer=tf.keras.regularizers.___(0.001)),
    tf.keras.layers.Dropout(rate=0.2), 
    tf.keras.layers.Dense(512, activation = 'relu', kernel_regularizer=tf.keras.regularizers.___(0.001)),
    tf.keras.layers.Dropout(rate=0.2), 
    tf.keras.layers.Dense(256, activation = 'relu', kernel_regularizer=tf.keras.regularizers.___(0.001)),
    tf.keras.layers.Dropout(rate=0.2), 
    tf.keras.layers.Dense(128, activation = 'relu', kernel_regularizer=tf.keras.regularizers.____(0.001)),
    tf.keras.layers.Dropout(rate=0.2), 
    tf.keras.layers.Dense(3, activation='softmax')
])

In [ ]:

lr = 0.0001
model_adv.compile(loss = 'categorical_crossentropy',
              optimizer = tf.keras.optimizers.Adam(learning_rate = lr),
              metrics=['accuracy']
)
early_stopping = EarlyStopping(monitor = 'val_loss', min_delta = 0, 
                               patience = 150, mode = 'auto',
                               restore_best_weights=True)

In [ ]:
epoch = 1000
history = model_adv.fit(X_train, Y_train, 
                        epochs = epoch, 
                        batch_size = 16, 
                        verbose = 0, 
                        validation_data=(X_test, Y_test_cat),
                        callbacks=[early_stopping])

In [ ]:
plt.title('Loss')
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.ylabel('Loss')
plt.xlabel('Epochs')
plt.legend()
plt.show()

In [ ]:
pred = model_adv.predict(X_test)
pred_classes = np.argmax(pred,axis=1)
acc = evaluation(pred_classes, Y_test, model_name="개선된 MLP")

In [ ]:
acc_dict['Advanced'] = acc

In [ ]:
bar = plt.bar(acc_dict.keys(), acc_dict.values())
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2.0, height, '%.2f' % height, ha='center', va='bottom', size = 12)
plt.ylim(0, 110)
plt.show()   

모델의 성능을 향상시키는 다양한 기법이 있지만, 해당 기법이 언제나 모델의 성능을 향상시켜 주는 것은 아닙니다. 따라서 데이터의 상태나 기법 후 성능 변화를 지켜보며 최적의 모델을 구현해나가는 과정이 중요합니다.

---
## 9. AutoML

인공신경망 모델에는 모델의 구조, 활성화 함수, 학습률 등 튜닝해야 할 파라미터가 매우 많습니다.

파라미터를 어떻게 튜닝하는냐에 따라 성능이 크게 좌우되기도 합니다.

이러한 과정을 줄여줄 수 있는 방법이 AutoML 기술입니다.

AutoML을 실습 데이터에 적용해보겠습니다.

---
### 9.1 AutoML 적용

In [ ]:
from autokeras import StructuredDataClassifier

In [ ]:
auto_model = StructuredDataClassifier(max_trials=30, overwrite=True)

auto_model.fit(X_train, Y_train, validation_data=(X_test, Y_test_cat))

기존 학습된 모델보다 더 좋은 검증 정답률을 보이는 모델을 자동으로 찾았습니다.

In [ ]:
model = auto_model.export_model()
model.summary()

모델의 구조는 위와 같습니다.

AutoML의 알고리즘은 구현 방식에 따라 랜덤성에 크게 의존하기 때문에, 시도 횟수를 늘리는 것이 좋습니다.